# ♠ ♰ Ｓ ｐ ａ Ｄ ｅ Ｄ ♰ ♠
### **Spatiotemporal Deepfake Detection via Texture-Enhanced Multi-Attentional ResNeXt50-BiLSTM**
#### `[ PROTOCOL: GOOGLE COLAB / CLOUD GPU TRAINING & INFERENCE PIPELINE ]`

This notebook automates end-to-end cloud training for **SpaDeD** with:
1. **Google Drive Integration**: Loads compressed datasets from 2TB Google Drive directly into fast `/content/` NVMe memory.
2. **Automated Environment**: Installs dependencies and verifies CUDA GPU acceleration.
3. **5-Fold Cross-Validation**: Trains the 4-Head BAP ResNeXt50-BiLSTM model with Dual Regional Independence Loss ($\mathcal{L}_{RIL}$).
4. **Persistent Weight Checkpointing**: Continuously mirrors best model weights (`spaded_best_auc.pth`) to Google Drive.

## ♰ [0x01] GPU DIAGNOSTIC & ACCELERATION CHECK

In [ ]:
# Verify GPU allocated by Colab (T4 / V100 / A100)
!nvidia-smi

import torch
print(f"[+] PyTorch Version: {torch.__version__}")
print(f"[+] CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[+] GPU Name:        {torch.cuda.get_device_name(0)}")
    print(f"[+] GPU Total VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[!] WARNING: No GPU detected. Please go to Runtime -> Change runtime type -> Select T4/A100 GPU.")

## ♰ [0x02] MOUNT GOOGLE DRIVE & REPOSITORY SYNC

In [ ]:
from google.colab import drive
import os

# Mount 2TB Google Drive
drive.mount('/content/drive')

# Define persistent Drive directories
DRIVE_DATASET_DIR = "/content/drive/MyDrive/SpaDeD_Datasets"
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/SpaDeD_Checkpoints"
os.makedirs(DRIVE_DATASET_DIR, exist_ok=True)
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

print(f"[+] Dataset Storage:    {DRIVE_DATASET_DIR}")
print(f"[+] Checkpoint Storage: {DRIVE_CHECKPOINT_DIR}")

## ♰ [0x03] CLONE REPOSITORY & INSTALL DEPENDENCIES

In [ ]:
%cd /content
!git clone https://github.com/XenonyxBlaze/spaded.git || (cd /content/spaded && git pull)
%cd /content/spaded

!pip install -q -r requirements.txt
!pip install -q tqdm opencv-python scikit-learn
print("[+] Dependencies successfully installed.")

## ♰ [0x04] UNPACK DATASETS FROM DRIVE TO HIGH-SPEED LOCAL DISK

In [ ]:
import os
from pathlib import Path

local_data = Path("/content/spaded/data")
local_data.mkdir(parents=True, exist_ok=True)

# 1. Unzip FaceForensics++ if uploaded to Drive
ff_zip = Path(DRIVE_DATASET_DIR) / "FaceForensics_c23.zip"
if ff_zip.exists():
    print(f"[>] Extracting {ff_zip.name} to {local_data}...")
    !unzip -q -o "{str(ff_zip)}" -d "{str(local_data)}"

# 2. Unzip Celeb-DF-v2 if uploaded to Drive
celeb_zip = Path(DRIVE_DATASET_DIR) / "Celeb-DF-v2.zip"
if celeb_zip.exists():
    print(f"[>] Extracting {celeb_zip.name} to {local_data}...")
    !unzip -q -o "{str(celeb_zip)}" -d "{str(local_data)}"

# 3. Unzip DF40 if uploaded to Drive
df40_zip = Path(DRIVE_DATASET_DIR) / "DF40_all.zip"
if df40_zip.exists():
    print(f"[>] Extracting {df40_zip.name} to {local_data}...")
    !unzip -q -o "{str(df40_zip)}" -d "{str(local_data)}"

print("\n[+] Data preparation complete! Local folders:")
!ls -lh /content/spaded/data

## ♰ [0x05] FORWARD / BACKWARD TENSOR SANITY CHECK

In [ ]:
!python run_pipeline.py --mode verify

## ♰ [0x06] LAUNCH 5-FOLD CROSS-VALIDATION TRAINING
Checkpoints are automatically saved to your persistent **Google Drive** after every epoch.

In [ ]:
!python run_pipeline.py --mode train \
    --dataset data/FaceForensics++ \
    --epochs 15 \
    --batch_size 16 \
    --checkpoint_dir /content/drive/MyDrive/SpaDeD_Checkpoints

## ♰ [0x07] DF40 UNSEEN CROSS-FORGERY EVALUATION

In [ ]:
!python run_pipeline.py --mode eval \
    --dataset data/DF40 \
    --weights /content/drive/MyDrive/SpaDeD_Checkpoints/spaded_best_auc.pth

## ♰ [0x08] STATISTICAL SIGNIFICANCE TESTS (BONFERRONI & BENJAMINI-HOCHBERG FDR)

In [ ]:
!python run_pipeline.py --mode stats